# Validation Results Inspection

This notebook provides a visual inspection of the validation runs generated by `scripts/validate_time_averaged.py`.

In [ ]:
import os
import sys

os.environ["CUDA_VISIBLE_DEVICES"] = "6"
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "False"
sys.path.append("..")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from gyaradax.plot_utils import plot_flux_trace, plot_spectra
from gyaradax.params import load_config

In [ ]:
output_dirs = sorted(
    [d for d in os.listdir("..") if d.startswith("validation_outputs_")]
)
print(f"Found {len(output_dirs)} validation directories: {output_dirs}")

## 1. Flux Comparison
Comparing Gyaradax (solid) against GKW reference (dashed).

In [ ]:
for out_dir in output_dirs:
    path = os.path.join("..", out_dir)

    flux_path = os.path.join(path, "fluxes.npz")
    growth_path = os.path.join(path, "growth.npz")

    if not os.path.exists(flux_path) or not os.path.exists(growth_path):
        print(f"Skipping {out_dir}: missing .npz files")
        continue

    sim_flux = np.load(flux_path)["fluxes"]
    sim_time = np.load(growth_path)["time"]

    config_name = out_dir.replace("validation_outputs_", "")
    config_path = os.path.join("..", "configs", f"{config_name}.yaml")

    ref_time = None
    ref_fluxes = None

    if os.path.exists(config_path):
        cfg = load_config(config_path)
        ref_dir = cfg.run.data_dir

        ref_time_path = os.path.join(ref_dir, "time.dat")
        ref_flux_path = os.path.join(ref_dir, "fluxes.dat")

        if os.path.exists(ref_time_path) and os.path.exists(ref_flux_path):
            ref_time = np.loadtxt(ref_time_path)
            ref_fluxes = np.loadtxt(ref_flux_path).T

    # 3. Plot Fluxes
    fig = plot_flux_trace(
        sim_time,
        sim_flux.T[[1, 2]],
        ref_time=ref_time,
        ref_fluxes=ref_fluxes[[1, 2]],
        labels=["Heat", "Momentum"],
        title=f"Fluxes: {config_name}",
    )
    plt.show()

    # 4. Plot Spectra Comparison (Last dump)
    # Search for the last dump in validation directory
    last_step_files = sorted(
        [f for f in os.listdir(path) if f.startswith("step_") and f.endswith(".npz")]
    )
    if last_step_files:
        last_dump = np.load(os.path.join(path, last_step_files[-1]))
        phi_final = last_dump["phi"]

        # Load kx, ky from geometry (via config)
        if os.path.exists(config_path):
            from gyaradax import load_geometry

            geom = load_geometry(ref_dir)
            kx = np.asarray(geom["kxrh"])
            ky = np.asarray(geom["krho"])

            fig_spec = plot_spectra(
                kx, ky, phi_final, title=f"Saturated Spectra: {config_name}"
            )

## 2. Performance Metrics
Analyzing steps per second across the simulation blocks.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

for out_dir in output_dirs:
    perf_path = os.path.join("..", out_dir, "performance.npz")
    if os.path.exists(perf_path):
        perf = np.load(perf_path)
        if "time" in perf and "steps_per_sec" in perf:
            ax.plot(
                perf["time"],
                perf["steps_per_sec"],
                "o-",
                label=out_dir.replace("validation_outputs_", ""),
            )

ax.set_xlabel("Simulation Time $[v_{th}/R]$")
ax.set_ylabel("Throughput [steps/s]")
ax.set_title("Solver Performance")
ax.legend(frameon=False)
ax.grid(True, linestyle=":", alpha=0.5)
plt.show()